# 新疆多级分类模型训练
本Notebook实现：
1. 先对大类（Formation）进行分类。
2. 再在每个大类下，训练小类（Alliance/num）分类模型。

In [1]:
# 导入所需库
import pandas as pd
import numpy as np
import tensorflow as tf
from tensorflow.keras import regularizers
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=DeprecationWarning)
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # 只显示error

d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:516: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint8 = np.dtype([("qint8", np.int8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:517: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_quint8 = np.dtype([("quint8", np.uint8, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\site-packages\tensorflow\python\framework\dtypes.py:518: FutureWarning: Passing (type, 1) or '1type' as a synonym of type is deprecated; in a future version of numpy, it will be understood as (type, (1,)) / '(1,)type'.
  _np_qint16 = np.dtype([("qint16", np.int16, 1)])
d:\ProgramData\Anaconda3\envs\tensorflow210\lib\s

## 读取与预处理数据（统一切分训练集和测试集）

In [2]:
# 读取训练数据和类别映射文件
train_data = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2.csv')
class_mapping = pd.read_csv(r'F:/TensorFlow/xinjiang/traindata20250626_2_class_mapping.csv')

# 去除Index列
if 'Index' in train_data.columns:
    train_data = train_data.drop(columns=['Index'])

# 合并class_mapping，将Alliance映射为num和Formation
train_data = train_data.merge(class_mapping[['Alliance', 'num', 'Formation']], on='Alliance', how='left')

# 特征、标签准备
feature_cols = [col for col in train_data.columns if col not in ['Alliance', 'num', 'Formation']]
X = train_data[feature_cols]
num_labels = train_data['num']
formation_labels = train_data['Formation']

# 编码大类标签
formation_encoder = LabelEncoder()
formation_y = formation_encoder.fit_transform(formation_labels)

# 统一切分训练集和测试集
X_train, X_test, num_train, num_test, formation_train, formation_test, formation_y_train, formation_y_test = train_test_split(
    X, num_labels, formation_labels, formation_y, test_size=0.2, random_state=42, shuffle=True
)

## 构建神经网络与损失函数

In [3]:
def build_model(input_dim, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(256, activation='relu', input_dim=input_dim, kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(64, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(32, activation='relu', kernel_regularizer=regularizers.l2(0.001)),
        tf.keras.layers.BatchNormalization(),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ])
    return model

def multi_category_focal_loss2(gamma=2., alpha=.25, class_weights=None):
    epsilon = 1.e-7
    gamma = float(gamma)
    alpha = tf.constant(alpha, dtype=tf.float32)
    if class_weights is not None:
        weights = np.array([class_weights.get(i, 1.0) for i in range(len(class_weights))])
        class_weights_tf = tf.constant(weights, dtype=tf.float32)
    else:
        class_weights_tf = None
    def focal_loss_fixed(y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.clip_by_value(y_pred, epsilon, 1. - epsilon)
        alpha_t = y_true * alpha + (1 - y_true) * (1 - alpha)
        y_t = y_true * y_pred + (1 - y_true) * (1 - y_pred)
        ce = -tf.math.log(y_t)
        weight = tf.pow(1. - y_t, gamma)
        fl = alpha_t * weight * ce
        if class_weights_tf is not None:
            fl = fl * class_weights_tf
        return tf.reduce_mean(fl)
    return focal_loss_fixed

def train_and_evaluate(x_train, y_train, num_classes, verbose=0):
    y_train_cat = tf.keras.utils.to_categorical(y_train, num_classes=num_classes)
    class_weights_array = compute_class_weight(class_weight='balanced', classes=np.arange(num_classes), y=y_train)
    class_weights = dict(enumerate(class_weights_array))
    model = build_model(x_train.shape[1], num_classes)
    optimizer = tf.keras.optimizers.Adam(learning_rate=1e-3)
    model.compile(loss=multi_category_focal_loss2(alpha=0.25, gamma=2, class_weights=class_weights),
                  optimizer=optimizer, metrics=['accuracy'])
    early_stopping = EarlyStopping(monitor='loss', patience=30, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='loss', factor=0.5, patience=10, verbose=0)
    history = model.fit(x_train, y_train_cat, epochs=500, batch_size=64, verbose=verbose, callbacks=[reduce_lr, early_stopping])
    n_epochs = len(history.history['loss'])
    final_loss = history.history['loss'][-1]
    final_acc = history.history['acc'][-1]
    print(f"训练轮数: {n_epochs}, 最终训练精度: {final_acc:.4f}, 最终训练损失: {final_loss:.4f}")
    return model

## 训练分类模型

In [4]:
# 训练大类模型（直接用X_train, formation_y_train）
num_formation_classes = len(np.unique(formation_y_train))
print(f"大类（Formation）类别数: {num_formation_classes}")
formation_model = train_and_evaluate(X_train, formation_y_train, num_formation_classes)
print('大类训练结束')

# 针对每个大类，训练小类（num）分类模型（直接用X_train, num_train, formation_y_train）
formation_list = formation_encoder.classes_
small_class_models = {}
for idx, formation in enumerate(formation_list):
    mask = (formation_y_train == idx)
    X_sub = X_train[mask]
    y_sub = num_train[mask]
    num_encoder = LabelEncoder()
    y_sub_encoded = num_encoder.fit_transform(y_sub)
    n_classes = len(np.unique(y_sub_encoded))
    if n_classes == 1:
        print(f"大类[{formation}] 中只有一个小类，无需训练模型，直接返回该类别")
        continue
    print(f"大类[{formation}]，中的小类数: {n_classes}")
    model = train_and_evaluate(X_sub, y_sub_encoded, n_classes)
    # 可选：保存模型、编码器等
    small_class_models[formation] = (model, num_encoder)
print('小类训练结束')

大类（Formation）类别数: 9
Instructions for updating:
Call initializer instance with the dtype argument instead of passing it to the constructor
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
Instructions for updating:
Use tf.where in 2.0, which has the same broadcast rule as np.where
训练轮数: 322, 最终训练精度: 0.7221, 最终训练损失: 0.0222
大类训练结束
大类[丛生草类草原]，中的小类数: 7
训练轮数: 322, 最终训练精度: 0.7221, 最终训练损失: 0.0222
大类训练结束
大类[丛生草类草原]，中的小类数: 7
训练轮数: 500, 最终训练精度: 0.9735, 最终训练损失: 0.0280
大类[丛生草类草甸]，中的小类数: 3
训练轮数: 500, 最终训练精度: 0.9735, 最终训练损失: 0.0280
大类[丛生草类草甸]，中的小类数: 3
训练轮数: 189, 最终训练精度: 0.8936, 最终训练损失: 0.3948
大类[农业植被] 中只有一个小类，无需训练模型，直接返回该类别
大类[半乔木与灌木荒漠]，中的小类数: 3
训练轮数: 189, 最终训练精度: 0.8936, 最终训练损失: 0.3948
大类[农业植被] 中只有一个小类，无需训练模型，直接返回该类别
大类[半乔木与灌木荒漠]，中的小类数: 3
训练轮数: 215, 最终训练精度: 0.8828, 最终训练损失: 0.3302
大类[半灌木与草本荒漠]，中的小类数: 12
训练轮数: 215, 最终训练精度: 0.8828, 最终训练损失: 0.3302
大类[半灌木与草本荒漠]，中的小类数: 12
训练轮数: 328, 最终训练精度: 0.8552, 最终训练损失: 0.0156
大类[半灌木草原] 中只有一个小类，无需训练模型，直接返回该类别
大类[杂类草草原] 中只有一个

### 预测

In [5]:
# 大类分类器对测试集进行预测
formation_pred_prob = formation_model.predict(X_test)
formation_pred = np.argmax(formation_pred_prob, axis=1)

acc = accuracy_score(formation_y_test, formation_pred)
prec = precision_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
rec = recall_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
f1 = f1_score(formation_y_test, formation_pred, average='weighted', zero_division=0)
print(f"植被型: accuracy: {acc:.4f}, precision: {prec:.4f}, recall: {rec:.4f}, f1: {f1:.4f}")

# 基于大类预测结果，对小类进行预测
num_pred = []
for i in range(len(X_test)):
    x_row = X_test.iloc[[i]]
    # 先预测大类
    formation_pred_prob = formation_model.predict(x_row)
    formation_pred_idx = np.argmax(formation_pred_prob, axis=1)[0]
    formation_name = formation_encoder.classes_[formation_pred_idx]
    # 如果该大类下只有一个小类，直接返回该小类
    mask = (formation_y_train == formation_pred_idx)
    y_sub = num_train[mask]
    if len(np.unique(y_sub)) == 1:
        num_pred.append(np.unique(y_sub)[0])
        continue
    # 用对应小类分类器预测
    model, num_encoder = small_class_models[formation_name]
    num_pred_prob = model.predict(x_row)
    num_pred_idx = np.argmax(num_pred_prob, axis=1)[0]
    num_pred_value = num_encoder.inverse_transform([num_pred_idx])[0]
    num_pred.append(num_pred_value)

# 计算num的四项指标
acc = accuracy_score(num_test, num_pred)
prec = precision_score(num_test, num_pred, average='weighted', zero_division=0)
rec = recall_score(num_test, num_pred, average='weighted', zero_division=0)
f1 = f1_score(num_test, num_pred, average='weighted', zero_division=0)
print(f"群系：accuracy: {acc:.4f}, precision: {prec:.4f}, recall: {rec:.4f}, f1: {f1:.4f}")

植被型: accuracy: 0.6703, precision: 0.6777, recall: 0.6703, f1: 0.6715
群系：accuracy: 0.5054, precision: 0.5387, recall: 0.5054, f1: 0.4864
群系：accuracy: 0.5054, precision: 0.5387, recall: 0.5054, f1: 0.4864
